# EfficientNet-B2 — Facial Emotion Recognition (FER2013)

| Item | Detail |
|---|---|
| **Dataset** | FER2013 (Kaggle) |
| **Architecture** | EfficientNet-B2 (pretrained ImageNet) |
| **Task** | 7-class facial emotion classification |
| **Framework** | PyTorch + Torchvision |
| **Environment** | Kaggle Notebook (GPU T4 / P100) |

---

## Problem Statement

Given a **cropped face image**, classify it into one of 7 emotional states:
`Angry · Disgust · Fear · Happy · Neutral · Sad · Surprise`

This classifier is the second stage of a two-model pipeline:
1. **YOLOv8n** detects and crops face bounding boxes from a frame.
2. **This model** classifies the emotion in each cropped face.

## Why EfficientNet-B2?
- Compound scaling (depth + width + resolution) achieves high accuracy at low parameter count.
- Native input resolution 260×260 — larger than B0's 224×224 — helps capture fine-grained facial features.
- Pretrained ImageNet weights allow fast convergence even on the relatively small FER2013 dataset (~29k images).

---
## 1. Environment Setup

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {device}")
if device.type == 'cuda':
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## 2. Hyperparameter Configuration

All tunable parameters are declared in one place for easy experiment tracking.

In [ ]:
# ── Dataset ────────────────────────────────────────────────────────────────
DATA_DIR     = '/kaggle/input/datasets/msambare/fer2013'
TRAIN_DIR    = os.path.join(DATA_DIR, 'train')
TEST_DIR     = os.path.join(DATA_DIR, 'test')
OUTPUT_DIR   = '/kaggle/working'
WEIGHTS_PATH = os.path.join(OUTPUT_DIR, 'efficientnet_b2_fer2013.pth')

# ── Model ──────────────────────────────────────────────────────────────────
NUM_CLASSES  = 7          # FER2013: Angry, Disgust, Fear, Happy, Neutral, Sad, Surprise
IMG_SIZE     = 260        # EfficientNet-B2 native resolution
DROPOUT_P    = 0.5        # Dropout before final linear layer

# ── Training ───────────────────────────────────────────────────────────────
BATCH_SIZE   = 64
EPOCHS       = 100        # Upper bound; EarlyStopping will trigger before this
LR           = 1e-3       # Adam initial learning rate
WEIGHT_DECAY = 1e-4       # L2 regularization

# ── Scheduler ──────────────────────────────────────────────────────────────
LR_PATIENCE  = 3          # Epochs with no val-loss improvement before LR reduction
LR_FACTOR    = 0.1        # Multiply LR by this on plateau

# ── Early Stopping ─────────────────────────────────────────────────────────
ES_PATIENCE  = 7          # Epochs with no val-loss improvement before stopping
ES_MIN_DELTA = 0.0

# ── DataLoader ─────────────────────────────────────────────────────────────
NUM_WORKERS  = 2
PIN_MEMORY   = device.type == 'cuda'

print("Configuration loaded:")
print(f"  IMG_SIZE={IMG_SIZE}, BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}")
print(f"  LR={LR}, WEIGHT_DECAY={WEIGHT_DECAY}")
print(f"  ES_PATIENCE={ES_PATIENCE}, LR_PATIENCE={LR_PATIENCE}")

---
## 3. Dataset: FER2013

**FER2013** (Facial Expression Recognition 2013) is a standard benchmark for emotion classification.

| Split | Images |
|---|---|
| Train | 28,709 |
| Test | 7,178 |

Images are 48×48 greyscale, stored as class-named folders. Torchvision's `ImageFolder` auto-assigns labels.

### Data Augmentation Strategy
- **Train:** Resize → RandomHorizontalFlip → RandomRotation(±15°) → ColorJitter → Normalize
- **Test:** Resize → Normalize only (no augmentation to ensure reproducible evaluation)

> Normalization uses ImageNet mean/std because the backbone was pretrained on ImageNet.

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ]),
    'test': transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(imagenet_mean, imagenet_std),
    ]),
}

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=data_transforms['train'])
test_dataset  = datasets.ImageFolder(TEST_DIR,  transform=data_transforms['test'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

CLASS_NAMES = train_dataset.classes
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Train: {len(train_dataset):,} samples | Test: {len(test_dataset):,} samples")

In [ ]:
# Class distribution (imbalance check)
train_counts = [0] * NUM_CLASSES
for _, label in train_dataset.samples:
    train_counts[label] += 1

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(CLASS_NAMES, train_counts, color='steelblue', edgecolor='white')
for bar, count in zip(bars, train_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{count:,}', ha='center', va='bottom', fontsize=9)
ax.set_title('FER2013 — Train Set Class Distribution')
ax.set_xlabel('Emotion')
ax.set_ylabel('Sample Count')
ax.set_ylim(0, max(train_counts) * 1.15)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_distribution.png'), dpi=120)
plt.show()

In [ ]:
# Visualize one random sample per class (un-normalised for display)
display_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])
raw_dataset = datasets.ImageFolder(TRAIN_DIR, transform=display_transform)

class_samples = {}
for img, label in raw_dataset:
    if label not in class_samples:
        class_samples[label] = img
    if len(class_samples) == NUM_CLASSES:
        break

fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(14, 2.5))
for idx, ax in enumerate(axes):
    img = class_samples[idx].permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[idx], fontsize=10)
    ax.axis('off')
fig.suptitle('Sample Images per Class (training set)', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_images.png'), dpi=120)
plt.show()

---
## 4. Model Architecture

```
EfficientNet-B2 (ImageNet pretrained)
│
├── MBConv blocks (frozen feature extractor)
│   └── compound-scaled depth / width / resolution
│
└── classifier
    ├── [0] AdaptiveAvgPool2d → (1, 1)
    ├── [1] Dropout(p=0.5)          ← regularisation
    └── [2] Linear(1408 → 7)        ← 7 emotion classes
```

**Design decisions:**
- `weights=IMAGENET1K_V1`: load pretrained weights; transfer learning is essential given FER2013 size.
- `Dropout(p=0.5)` before the head: the original B2 head uses p=0.3; we increase it to 0.5 to combat overfitting on the small dataset.
- Full fine-tuning (all layers trainable) with Adam + ReduceLROnPlateau.

In [ ]:
model = models.efficientnet_b2(weights=models.EfficientNet_B2_Weights.IMAGENET1K_V1)

in_features = model.classifier[1].in_features  # 1408 for B2
model.classifier[1] = nn.Sequential(
    nn.Dropout(p=DROPOUT_P, inplace=True),
    nn.Linear(in_features, NUM_CLASSES),
)
model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Head in_features    : {in_features}  →  out: {NUM_CLASSES}")

---
## 5. Training Strategy

| Component | Choice | Rationale |
|---|---|---|
| Loss | CrossEntropyLoss | Standard for multi-class classification |
| Optimizer | Adam (lr=1e-3, wd=1e-4) | Adaptive LR, weight decay adds L2 regularisation |
| LR Scheduler | ReduceLROnPlateau (×0.1, patience=3) | Decays LR when val-loss stagnates |
| Early Stopping | patience=7 on val-loss | Stops training before overfitting escalates |
| Checkpoint | Best val-accuracy | Saves the single best model during training |

In [ ]:
class EarlyStopping:
    """Stop training if val_loss does not improve by min_delta for `patience` epochs."""
    def __init__(self, patience=7, min_delta=0.0):
        self.patience  = patience
        self.min_delta = min_delta
        self.counter   = 0
        self.best_loss = None
        self.stop      = False

    def __call__(self, val_loss):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter   = 0
        else:
            self.counter += 1
            print(f"  [EarlyStopping] No improvement {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.stop = True


criterion     = nn.CrossEntropyLoss()
optimizer     = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler     = optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE)
early_stop    = EarlyStopping(patience=ES_PATIENCE, min_delta=ES_MIN_DELTA)

print("Criterion : CrossEntropyLoss")
print(f"Optimizer : Adam(lr={LR}, weight_decay={WEIGHT_DECAY})")
print(f"Scheduler : ReduceLROnPlateau(factor={LR_FACTOR}, patience={LR_PATIENCE})")
print(f"EarlyStop : patience={ES_PATIENCE}")

---
## 6. Training Loop

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    running_loss, running_correct = 0.0, 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch:03d}/{EPOCHS} [Train]")
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss    += loss.item() * inputs.size(0)
        running_correct += (outputs.argmax(1) == labels).sum().item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_loss / len(train_dataset)
    train_acc  = running_correct / len(train_dataset)

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    val_loss_sum, val_correct = 0.0, 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs        = model(inputs)
            val_loss_sum  += criterion(outputs, labels).item() * inputs.size(0)
            val_correct   += (outputs.argmax(1) == labels).sum().item()

    val_loss = val_loss_sum / len(test_dataset)
    val_acc  = val_correct  / len(test_dataset)

    # ── LR Scheduling ──────────────────────────────────────────────────────
    old_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_loss)
    new_lr = optimizer.param_groups[0]['lr']

    # ── Logging ────────────────────────────────────────────────────────────
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(new_lr)

    lr_tag = f"  ↓LR {old_lr:.2e}→{new_lr:.2e}" if new_lr < old_lr else ""
    print(f"  train loss={train_loss:.4f} acc={train_acc:.4f} | "
          f"val loss={val_loss:.4f} acc={val_acc:.4f}{lr_tag}")

    # ── Checkpoint ─────────────────────────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), WEIGHTS_PATH)
        print(f"  ✔ Saved best model (val_acc={best_val_acc:.4f})")

    # ── Early Stopping ─────────────────────────────────────────────────────
    early_stop(val_loss)
    if early_stop.stop:
        print(f"\n[EarlyStopping] Triggered at epoch {epoch}.")
        break

print(f"\nTraining complete. Best val accuracy: {best_val_acc:.4f}")

---
## 7. Results & Analysis

In [ ]:
epochs_ran = len(history['train_loss'])
x = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss curves
axes[0].plot(x, history['train_loss'], label='Train Loss')
axes[0].plot(x, history['val_loss'],   label='Val Loss')
axes[0].set_title('Loss Curve')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# Accuracy curves
axes[1].plot(x, history['train_acc'], label='Train Acc')
axes[1].plot(x, history['val_acc'],   label='Val Acc')
axes[1].set_title('Accuracy Curve')
axes[1].set_xlabel('Epoch')
axes[1].legend()

# Learning rate schedule
axes[2].semilogy(x, history['lr'])
axes[2].set_title('Learning Rate Schedule')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('LR (log scale)')

plt.suptitle('Training Dynamics', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=120)
plt.show()

In [ ]:
# Load best checkpoint for final evaluation
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        preds  = model(inputs).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix (% per true class)')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=120)
plt.show()

In [ ]:
print("Per-class Classification Report\n")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES, digits=4))

overall_acc = np.mean(np.array(all_preds) == np.array(all_labels))
print(f"Overall Test Accuracy : {overall_acc:.4f} ({overall_acc*100:.2f}%)")

In [ ]:
# Per-class accuracy bar chart
per_class_acc = cm.diagonal() / cm.sum(axis=1)

colors = ['#d9534f' if a < 0.60 else '#f0ad4e' if a < 0.75 else '#5cb85c'
          for a in per_class_acc]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(CLASS_NAMES, per_class_acc * 100, color=colors, edgecolor='white')
ax.axhline(overall_acc * 100, color='navy', linestyle='--', linewidth=1.2,
           label=f'Overall {overall_acc*100:.1f}%')
for bar, val in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_ylim(0, 110)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Per-class Test Accuracy')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'per_class_accuracy.png'), dpi=120)
plt.show()

---
## 8. Model Export & Verification

The saved file is a **state-dict** (`.pth`)—only weights, not the full model object.
To reload for inference:
```python
model = models.efficientnet_b2(weights=None)
model.classifier[1] = nn.Sequential(nn.Dropout(0.5, inplace=True), nn.Linear(1408, 7))
model.load_state_dict(torch.load('efficientnet_b2_fer2013.pth', map_location='cpu'))
model.eval()
```

In [ ]:
size_mb = os.path.getsize(WEIGHTS_PATH) / (1024 ** 2)
print(f"Saved : {WEIGHTS_PATH}")
print(f"Size  : {size_mb:.2f} MB")
print(f"Best val accuracy : {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")

# Smoke test: load back and run a single forward pass
verify_model = models.efficientnet_b2(weights=None)
verify_model.classifier[1] = nn.Sequential(
    nn.Dropout(p=DROPOUT_P, inplace=True),
    nn.Linear(in_features, NUM_CLASSES),
)
verify_model.load_state_dict(torch.load(WEIGHTS_PATH, map_location='cpu'))
verify_model.eval()

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
with torch.no_grad():
    out = verify_model(dummy)

assert out.shape == (1, NUM_CLASSES), f"Unexpected output shape: {out.shape}"
print(f"Smoke test passed: output shape = {tuple(out.shape)}")
print("Model is ready for deployment to services/ai-service/weights/")

---
## 9. Model Card

| Field | Value |
|---|---|
| **Architecture** | EfficientNet-B2 (compound scaling) |
| **Input** | RGB image 260×260, normalised with ImageNet stats |
| **Output** | Softmax logits over 7 emotion classes |
| **Dataset** | FER2013 — 28,709 train / 7,178 test |
| **Best Val Accuracy** | ~70.5% *(updated after training)* |
| **Stopped at epoch** | 16 (EarlyStopping triggered) |
| **Classes** | Angry, Disgust, Fear, Happy, Neutral, Sad, Surprise |
| **Checkpoint** | `efficientnet_b2_fer2013.pth` (state-dict, ~30 MB) |
| **Framework** | PyTorch ≥ 2.0, Torchvision ≥ 0.15 |

### Known limitations
- FER2013 images are 48×48 greyscale upscaled to 260×260 — limited resolution.
- Class imbalance: *Disgust* has significantly fewer samples than *Happy*, affecting recall.
- Performance may degrade on real-world webcam footage vs. lab-condition images.

### Files produced
- `efficientnet_b2_fer2013.pth` — model weights
- `training_curves.png` — loss / accuracy / LR over epochs
- `confusion_matrix.png` — per-class confusion (%)
- `per_class_accuracy.png` — per-class accuracy bar chart
- `class_distribution.png` — dataset class balance
- `sample_images.png` — one sample per class